# LLM Observability Hackathon

You are the investigator. Each activity produces traces in your local Langfuse
project; open Langfuse at `http://localhost:3000` after the compose stack is up.

## Setup

Required environment variables:

- `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, `LANGFUSE_BASE_URL`
- `OPENAI_BASE_URL`, `OPENAI_API_KEY`, `OPENAI_MODEL`

The notebook fails loudly when any variable is missing. Model calls use
`temperature=0` and fixed inputs; trace ids and the run id vary per run.

In [1]:
import os, json, re, time, uuid
from langfuse import Langfuse, propagate_attributes
from langfuse.openai import OpenAI

REQUIRED = ["LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "LANGFUSE_BASE_URL",
            "OPENAI_BASE_URL", "OPENAI_API_KEY", "OPENAI_MODEL"]
missing = [k for k in REQUIRED if not os.environ.get(k)]
if missing:
    raise RuntimeError("Missing environment variables: " + ", ".join(missing))

In [2]:
langfuse = Langfuse()
try:
    authenticated = langfuse.auth_check()
except Exception as exc:
    raise RuntimeError(f"Langfuse authentication failed for {os.environ['LANGFUSE_BASE_URL']}: {exc}") from exc
if not authenticated:
    raise RuntimeError(f"Langfuse authentication failed for {os.environ['LANGFUSE_BASE_URL']}")
client = OpenAI(base_url=os.environ["OPENAI_BASE_URL"], api_key=os.environ["OPENAI_API_KEY"])
MODEL = os.environ["OPENAI_MODEL"]
RUN_ID = uuid.uuid4().hex[:8]

def show_trace(trace_id):
    url = langfuse.get_trace_url(trace_id=trace_id)
    print(f"trace: {url or trace_id}")
    return url or trace_id

print("run id:", RUN_ID)

run id: 26952240


## 1. Activity 1 — RAG with and without filter

**Goal:** Compare a RAG answer produced with unfiltered retrieval against one produced with a tier filter.

**Expected observation:** The unfiltered run retrieves a policy for another tier and answers from it; the filtered run retrieves the Basic policy.

**Deliverable:** Both trace URLs, the retrieved document ids, and the answer difference.

The customer profile says Basic; the question itself does not mention the tier. The retrieval filter is what is supposed to supply it.

In [3]:
from IPython.display import Markdown, display

CORPUS = json.load(open("data/travel_corpus.json", encoding="utf-8"))

def load_corpus(path):
    with open(path, encoding="utf-8") as handle:
        return json.load(handle)

def retrieve(query, tier=None):
    documents = CORPUS["documents"]
    if tier is not None:
        documents = [doc for doc in documents if doc["tier"] == tier]
    terms = query.lower().split()
    ranked = sorted(
        ((sum(term in doc["text"].lower() for term in terms), index, doc)
         for index, doc in enumerate(documents)),
        key=lambda row: (-row[0], row[1]),
    )
    score, _, doc = ranked[0]
    return {"id": doc["id"], "tier": doc["tier"], "title": doc["title"], "text": doc["text"], "score": score}

def run_rag(customer_message, tier, variant):
    trace_name = f"activity-1-rag-{variant}"
    with langfuse.start_as_current_observation(name=trace_name, as_type="span") as root:
        with propagate_attributes(
            session_id=f"activity-1-{RUN_ID}",
            tags=["activity-1", variant],
            trace_name=trace_name,
        ):
            doc = retrieve("checked bag allowance", tier)
            with langfuse.start_as_current_observation(
                name="activity-1-retrieval",
                as_type="retriever",
                input={"query": "checked bag allowance",
                       "filter": None if tier is None else {"tier": tier}},
                output={"doc_id": doc["id"], "tier": doc["tier"], "text": doc["text"]},
            ):
                pass
            messages = [
                {"role": "system", "content": "Answer using the retrieved policy document."},
                {"role": "user", "content": f"{customer_message}\n\nRetrieved policy:\n{doc['text']}"},
            ]
            started = time.time()
            response = client.chat.completions.create(
                model=MODEL, messages=messages, temperature=0, max_tokens=300,
                name="activity-1-generation",
            )
            latency = time.time() - started
            answer = response.choices[0].message.content
            trace_id = langfuse.get_current_trace_id()
            root.update(output={"answer": answer, "doc_id": doc["id"]})
    usage = response.usage
    return {
        "variant": variant,
        "answer": answer,
        "doc": doc,
        "input_tokens": getattr(usage, "prompt_tokens", None),
        "output_tokens": getattr(usage, "completion_tokens", None),
        "latency": latency,
        "trace_id": trace_id,
        "trace_url": langfuse.get_trace_url(trace_id=trace_id),
    }

customer = "How many checked bags do I get on my booking?"
runs = [run_rag(customer, None, "no-filter"), run_rag(customer, "basic", "filter")]

rows = ["| variant | retrieved doc | answer | input tokens | output tokens | latency (s) | trace |",
        "|---|---|---|---|---|---|---|"]
for run in runs:
    doc = run["doc"]
    answer = (run["answer"] or "").replace("\n", " ").replace("|", "\\|")
    rows.append(f"| {run['variant']} | {doc['id']} ({doc['tier']}) | {answer} | {run['input_tokens']} | {run['output_tokens']} | {run['latency']:.1f} | {run['trace_url']} |")
display(Markdown("\n".join(rows)))

| variant | retrieved doc | answer | input tokens | output tokens | latency (s) | trace |
|---|---|---|---|---|---|---|
| no-filter | baggage-gold (gold) | You get **one 23 kg checked bag included** with the Gold fare. A second checked bag costs **60 EUR**. | 71 | 92 | 1.6 | http://localhost:3000/project/langfuse-observability/traces/d22948d02be02baea1b890931bbfd45e |
| filter | baggage-basic (basic) | You don’t get any checked bags included with the Basic fare. You can add a 23 kg checked bag for 45 EUR. | 82 | 103 | 1.6 | http://localhost:3000/project/langfuse-observability/traces/6f5f6904a051352e4d3831216eca57e7 |

## 2. Activity 2 — Output format

**Goal:** Compare an unconstrained prose answer with a constrained JSON answer for a downstream booking parser.

**Expected observation:** The prose answer fails validation; the constrained answer parses.

**Deliverable:** Validation results and trace URLs.

## 3. Activity 3 — Fix hallucination

**Goal:** Find the cause of a wrong baggage answer in the trace and fix the system prompt.

**Expected observation:** The broken run answers two checked bags while the retrieved Gold policy says one.

**Deliverable:** Root cause and the fixed answer.

## 4. Activity 4 — Debug tool calling

**Goal:** Diagnose a failing tool call from trace data and repair the tool.

**Expected observation:** The tool span shows the arguments and a KeyError; the fixed run returns the booking.

**Deliverable:** Failing span details and the repaired answer.

## 5. Wrap-up

Flush the Langfuse client so the last traces are delivered. Submit the trace URLs
and your root causes for Activities 3 and 4.

In [4]:
langfuse.flush()
print("flushed; run id:", RUN_ID)

flushed; run id: 26952240
